# Loading dataset and EDA

In [1]:
import pandas as pd
import numpy as np

In [2]:
recom_df = pd.read_csv('recom.csv')
recom_df

,Unnamed: 0,Main_ID,Transaction_ID,Date,Price,Code_Product,Amount,ItemKey
0,0,90fada91,264f7a69,2022-10-07 20:53:49.153,125.0,5002.0,1.0,5002.0
1,1,9006f9ac,45c7d853,2022-09-17 15:54:57.187,19.0,35012.0,1.0,NaN
2,2,32270891,61ad76dd,2022-11-28 13:51:55.667,141.0,5005.0,1.0,5005.0
3,3,97e03e47,41ee09f6,2022-09-12 16:20:22.110,4.5,35078.5,1.0,NaN
4,4,41949228,244fe6d8,2022-10-14 18:53:43.933,129.5,49291.5,5.0,NaN
...,...,...,...,...,...,...,...,...
49995,49995,bb127ffb,4e0eb5ab,2022-09-24 21:48:20.847,111.5,45004.0,4.0,45004.0
49996,49996,a8bc484a,c9946c16,2022-11-18 19:49:01.973,34.0,49292.0,1.0,NaN
49997,49997,c983862a,d1a35c5c,2022-11-24 20:02:43.023,178.0,5001.5,1.0,5001.5
49998,49998,8821da12,66f9b474,2022-11-06 13:07:01.423,26.0,49291.5,1.0,NaN


'Unnamed: 0' is likely an ETL artifact. We can delete it.

In [3]:
recom_df.drop('Unnamed: 0', axis=1, inplace=True)

I suspect that Code_Product and ItemKey are redundant, but ItemKey has nulls so I will have to check.

In [4]:
recom_df['mismatch'] = np.where(
    recom_df['Code_Product'].notna() &
    recom_df['ItemKey'].notna() &
    (recom_df['Code_Product'] != recom_df['ItemKey']),
    1,
    0
)

In [5]:
recom_df['mismatch'].value_counts()

mismatch
0    50000
Name: count, dtype: int64

As I thought, ItemKey is identical to Code_Product when not null, so we can drop it. I'll also rename Code_Product to better match the naming of other columns.

In [6]:
recom_df.drop('mismatch', axis=1, inplace=True)
recom_df.drop('ItemKey', axis=1, inplace=True)
recom_df.rename(columns={'Code_Product': 'Product_ID'}, inplace=True)
recom_df.rename(columns={'Main_ID': 'Customer_ID'}, inplace=True)

Quick EDA.

In [7]:
recom_df['Customer_ID'].nunique()

28514

In [8]:
recom_df['Transaction_ID'].nunique()

48403

In [9]:
recom_df['Product_ID'].nunique()

333

In [10]:
recom_df['Amount'].value_counts()

Amount
1.0     41695
2.0      6369
3.0      1279
4.0       415
5.0       130
6.0        56
7.0        16
10.0       11
8.0         7
20.0        4
13.0        4
18.0        3
9.0         2
54.0        1
33.0        1
19.0        1
41.0        1
17.0        1
28.0        1
14.0        1
16.0        1
12.0        1
Name: count, dtype: int64

In [11]:
transac_by_user = recom_df.groupby('Customer_ID')['Transaction_ID'].value_counts().sort_values(ascending=False)
transac_by_user.value_counts()

count
1    46859
2     1491
3       53
Name: count, dtype: int64

In [12]:
recom_df

,Customer_ID,Transaction_ID,Date,Price,Product_ID,Amount
0,90fada91,264f7a69,2022-10-07 20:53:49.153,125.0,5002.0,1.0
1,9006f9ac,45c7d853,2022-09-17 15:54:57.187,19.0,35012.0,1.0
2,32270891,61ad76dd,2022-11-28 13:51:55.667,141.0,5005.0,1.0
3,97e03e47,41ee09f6,2022-09-12 16:20:22.110,4.5,35078.5,1.0
4,41949228,244fe6d8,2022-10-14 18:53:43.933,129.5,49291.5,5.0
...,...,...,...,...,...,...
49995,bb127ffb,4e0eb5ab,2022-09-24 21:48:20.847,111.5,45004.0,4.0
49996,a8bc484a,c9946c16,2022-11-18 19:49:01.973,34.0,49292.0,1.0
49997,c983862a,d1a35c5c,2022-11-24 20:02:43.023,178.0,5001.5,1.0
49998,8821da12,66f9b474,2022-11-06 13:07:01.423,26.0,49291.5,1.0


# Aggregates and new dataframes

In [26]:
recom_df_with_aggs = recom_df.copy()

# unique products per transaction
recom_df_with_aggs['unique_products_per_transac'] = (recom_df.groupby('Transaction_ID')['Product_ID'].transform('nunique'))

# total items per transaction
recom_df_with_aggs['total_items_per_transac'] = (recom_df.groupby('Transaction_ID')['Product_ID'].transform('count'))

# total spent per transaction
recom_df_with_aggs['total_spent_per_transac'] = (recom_df.groupby('Transaction_ID')['Price'].transform('sum'))

# =====================
# =====================

# unique products per customer
recom_df_with_aggs['unique_products_per_customer'] = (recom_df.groupby('Customer_ID')['Product_ID'].transform('nunique'))

# total items per customer
recom_df_with_aggs['total_items_per_customer'] = (recom_df.groupby('Customer_ID')['Product_ID'].transform('count'))

# total spent per customer
recom_df_with_aggs['total_spent_per_customer'] = (recom_df.groupby('Customer_ID')['Price'].transform('sum'))

# total transactions per customer
recom_df_with_aggs['transac_per_customer'] = (recom_df.groupby('Customer_ID')['Transaction_ID'].transform('nunique'))

# =====================
# =====================

# unique customers per product
recom_df_with_aggs['unique_customers_per_product'] = (recom_df.groupby('Product_ID')['Customer_ID'].transform('nunique'))

# total quantity sold per product
recom_df_with_aggs['total_sold_per_product'] = (recom_df.groupby('Product_ID')['Amount'].transform('sum'))

# total spent per product
recom_df_with_aggs['total_revenue_per_product'] = recom_df_with_aggs['total_sold_per_product'] * recom_df_with_aggs['Price']

In [27]:
recom_df_with_aggs

,Customer_ID,Transaction_ID,Date,Price,Product_ID,Amount,unique_products_per_transac,total_items_per_transac,total_spent_per_transac,unique_products_per_customer,total_items_per_customer,total_spent_per_customer,transac_per_customer,unique_customers_per_product,total_sold_per_product,total_revenue_per_product
0,90fada91,264f7a69,2022-10-07 20:53:49.153,125.0,5002.0,1.0,1,1,125.0,1,1,125.0,1,660,796.0,99500.0
1,9006f9ac,45c7d853,2022-09-17 15:54:57.187,19.0,35012.0,1.0,1,1,19.0,4,4,90.5,4,8,8.0,152.0
2,32270891,61ad76dd,2022-11-28 13:51:55.667,141.0,5005.0,1.0,1,1,141.0,8,9,825.0,9,44,68.0,9588.0
3,97e03e47,41ee09f6,2022-09-12 16:20:22.110,4.5,35078.5,1.0,1,1,4.5,1,1,4.5,1,132,189.0,850.5
4,41949228,244fe6d8,2022-10-14 18:53:43.933,129.5,49291.5,5.0,1,1,129.5,1,1,129.5,1,4127,5706.0,738927.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,bb127ffb,4e0eb5ab,2022-09-24 21:48:20.847,111.5,45004.0,4.0,1,1,111.5,3,4,469.0,4,2732,4507.0,502530.5
49996,a8bc484a,c9946c16,2022-11-18 19:49:01.973,34.0,49292.0,1.0,1,1,34.0,1,1,34.0,1,4270,6006.0,204204.0
49997,c983862a,d1a35c5c,2022-11-24 20:02:43.023,178.0,5001.5,1.0,1,1,178.0,12,16,2415.5,12,461,512.0,91136.0
49998,8821da12,66f9b474,2022-11-06 13:07:01.423,26.0,49291.5,1.0,1,1,26.0,1,1,26.0,1,4127,5706.0,148356.0


In [28]:
transactions_df = recom_df_with_aggs[['Transaction_ID', 'Customer_ID', 'Date', 'Price', 'Product_ID', 'Amount', 'unique_products_per_transac', 'total_items_per_transac', 'total_spent_per_transac']] 
transactions_df = transactions_df.drop_duplicates(subset=['Transaction_ID'], keep='first')
transactions_df

,Transaction_ID,Customer_ID,Date,Price,Product_ID,Amount,unique_products_per_transac,total_items_per_transac,total_spent_per_transac
0,264f7a69,90fada91,2022-10-07 20:53:49.153,125.0,5002.0,1.0,1,1,125.0
1,45c7d853,9006f9ac,2022-09-17 15:54:57.187,19.0,35012.0,1.0,1,1,19.0
2,61ad76dd,32270891,2022-11-28 13:51:55.667,141.0,5005.0,1.0,1,1,141.0
3,41ee09f6,97e03e47,2022-09-12 16:20:22.110,4.5,35078.5,1.0,1,1,4.5
4,244fe6d8,41949228,2022-10-14 18:53:43.933,129.5,49291.5,5.0,1,1,129.5
...,...,...,...,...,...,...,...,...,...
49995,4e0eb5ab,bb127ffb,2022-09-24 21:48:20.847,111.5,45004.0,4.0,1,1,111.5
49996,c9946c16,a8bc484a,2022-11-18 19:49:01.973,34.0,49292.0,1.0,1,1,34.0
49997,d1a35c5c,c983862a,2022-11-24 20:02:43.023,178.0,5001.5,1.0,1,1,178.0
49998,66f9b474,8821da12,2022-11-06 13:07:01.423,26.0,49291.5,1.0,1,1,26.0


In [29]:
products_df = recom_df_with_aggs[['Product_ID', 'Customer_ID', 'Price', 'unique_customers_per_product', 'total_sold_per_product', 'total_revenue_per_product']]
products_df = products_df.drop_duplicates(subset=['Product_ID'], keep='first')
products_df

,Product_ID,Customer_ID,Price,unique_customers_per_product,total_sold_per_product,total_revenue_per_product
0,5002.0,90fada91,125.0,660,796.0,99500.0
1,35012.0,9006f9ac,19.0,8,8.0,152.0
2,5005.0,32270891,141.0,44,68.0,9588.0
3,35078.5,97e03e47,4.5,132,189.0,850.5
4,49291.5,41949228,129.5,4127,5706.0,738927.0
...,...,...,...,...,...,...
44489,60051.0,a1fedd5e,48.0,1,3.0,144.0
45174,35072.0,9704c040,25.5,1,1.0,25.5
45448,20034.0,df4ce5ac,344.5,1,1.0,344.5
45861,48590.0,b4db2269,52.5,1,1.0,52.5


In [32]:
customers_df = recom_df_with_aggs[['Customer_ID', 'unique_products_per_customer', 'total_items_per_customer', 'transac_per_customer', 'total_spent_per_customer']]
customers_df = customers_df.drop_duplicates(subset=['Customer_ID'], keep='first')
customers_df

,Customer_ID,unique_products_per_customer,total_items_per_customer,transac_per_customer,total_spent_per_customer
0,90fada91,1,1,1,125.0
1,9006f9ac,4,4,4,90.5
2,32270891,8,9,9,825.0
3,97e03e47,1,1,1,4.5
4,41949228,1,1,1,129.5
...,...,...,...,...,...
49986,50fef10a,1,1,1,40.5
49987,db8a4949,1,1,1,18.5
49988,2316a392,1,1,1,50.5
49996,a8bc484a,1,1,1,34.0


In [31]:
customers_df.nlargest(10, 'total_spent_per_customer')

,Customer_ID,Date,Price,Product_ID,Amount,unique_products_per_customer,total_items_per_customer,transac_per_customer,total_spent_per_customer
1988,751131ee,2022-10-16 22:01:21.277,421.5,5001.0,2.0,23,51,39,28082.5
4290,c25c373a,2022-09-01 22:23:07.607,621.5,5004.5,1.0,17,25,19,19561.5
7656,31812c26,2022-08-31 21:39:30.243,767.0,5009.0,3.0,14,19,16,11399.5
1564,4493b2ac,2022-11-09 13:32:21.283,211.0,5025.0,1.0,15,35,33,6318.5
1239,0c87428f,2022-10-26 19:15:31.290,462.0,5025.0,1.0,13,19,17,6065.0
2268,d0778daf,2022-10-04 12:44:09.420,201.0,5009.0,2.0,23,39,37,5910.5
323,aa6d52d8,2022-11-03 19:14:03.873,106.5,45001.0,1.0,21,36,32,5652.5
689,734264e5,2022-10-10 20:30:54.350,319.5,30003.5,1.0,33,47,44,5189.5
563,07261a71,2022-09-30 18:45:02.367,224.5,48504.5,5.0,24,33,26,5058.0
936,744950ba,2022-11-23 17:04:44.123,1574.5,45001.5,10.0,6,6,4,4985.5
